# Non-Abelian Gauge Erasure: A Universal Structural Theorem

**Paper 3 — Technical Notebook**

This notebook implements and verifies the gauge erasure theorem:
non-Abelian gauge degrees of freedom are universally erased upon
hydrodynamization, while U(1) gauge structure survives as a 1-form
Goldstone boson (the photon).

The argument is purely algebraic, resting on:
1. Higher-form symmetries must be Abelian (codimension >1 operators commute)
2. Non-Abelian gauge theories have at most discrete center symmetries
3. Discrete symmetry breaking produces domain walls, not Goldstone bosons
4. Universal across standard, fracton, and holographic hydrodynamics

All physics functions imported from `src.gauge_erasure` (single source of truth).
Lean formalization: `GaugeErasure.lean`.

In [ ]:
# ════════════════════════════════════════════════════════════════════
# Setup: imports from src.gauge_erasure and src.core.visualizations
# ════════════════════════════════════════════════════════════════════
import os
import sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import plotly.graph_objects as go

from src.core.visualizations import (
    COLORS,
    FONT,
    TITLE_FONT,
    LAYOUT_TEMPLATE,
    apply_layout,
)  # Full palette (experiments + physics categories)

from src.gauge_erasure.erasure_theorem import (
    GaugeGroup, GaugeGroupType, HigherFormSymmetry, HydrodynamicFate,
    GaugeErasureResult, su, u1, so,
    is_abelian, center_subgroup, higher_form_symmetry_type,
    goldstone_or_domain_wall, survives_hydrodynamization,
    gauge_erasure_analysis, standard_model_analysis, n4_sym_analysis,
)

print("Gauge erasure module loaded from src.gauge_erasure (single source of truth).")
print(f"  Available group constructors: su(N), u1(), so(N)")
print(f"  Analysis functions: gauge_erasure_analysis, standard_model_analysis, n4_sym_analysis")

## 1. The Argument Chain

The gauge erasure theorem follows a chain of purely algebraic steps:

| Step | Statement | Key Input |
|------|-----------|----------|
| (i) | Hydrodynamic modes arise from spontaneously broken continuous symmetries | Goldstone theorem |
| (ii) | In gauge theories, the relevant symmetries are higher-form (p-form) symmetries | Gaiotto et al. (2015) |
| (iii) | For p >= 1, charged operators on codimension >1 submanifolds commute | Topology |
| (iv) | Therefore higher-form symmetry groups must be Abelian | (ii) + (iii) |
| (v) | Non-Abelian gauge theories have at most discrete Z_N center symmetry | Group theory |
| (vi) | Discrete symmetry breaking -> domain walls, not Goldstone bosons | SSB classification |
| (vii) | Therefore: no non-Abelian gauge hydrodynamic modes | (i) + (vi) |

**Contrast:** U(1) gauge theory has a continuous magnetic 1-form symmetry U(1)^{(1)}.
Its spontaneous breaking produces the photon as a 1-form Goldstone boson
(Grozdanov-Hofman-Iqbal photonization theorem).

In [ ]:
# ════════════════════════════════════════════════════════════════════
# Demonstrate the full argument chain for SU(3)
# ════════════════════════════════════════════════════════════════════
result_su3 = gauge_erasure_analysis(su(3))

print("GAUGE ERASURE ANALYSIS: SU(3)")
print("=" * 70)
print(f"  Group:              {result_su3.group.name}")
print(f"  Dimension:          {result_su3.group.dim}")
print(f"  Is Abelian:         {result_su3.is_abelian}")
print(f"  Center subgroup:    {result_su3.group.center}")
print(f"  Higher-form type:   {result_su3.higher_form_type.value}")
print(f"  Hydrodynamic fate:  {result_su3.fate.value}")
print(f"  Survives hydro:     {result_su3.survives}")
print()
print("Explanation:")
print(f"  {result_su3.explanation}")

## 2. Higher-Form Symmetry Classification

The critical insight (Gaiotto-Kapustin-Seiberg-Willett 2015): higher-form
symmetries are classified by the commutativity of their charged operators.
For p-form symmetries with p >= 1, the charged operators are supported on
submanifolds of codimension > 1 in space, so they necessarily commute.

**Consequence:** All higher-form symmetry groups must be Abelian.

This is the structural root of gauge erasure: non-Abelian gauge groups
cannot have continuous higher-form symmetries.

In [ ]:
# ════════════════════════════════════════════════════════════════════
# Higher-form symmetry classification for a range of gauge groups
# ════════════════════════════════════════════════════════════════════
groups = [
    ("U(1)", u1()),
    ("SU(2)", su(2)),
    ("SU(3)", su(3)),
    ("SU(5)", su(5)),
    ("SO(10)", so(10)),
]

print(f"{'Group':<10} {'Abelian?':<10} {'Center':<15} {'Higher-form type':<25} {'Continuous?'}")
print("-" * 75)
for label, g in groups:
    hf = higher_form_symmetry_type(g)
    continuous = (hf == HigherFormSymmetry.CONTINUOUS_1FORM)
    print(f"{label:<10} {str(g.is_abelian):<10} {g.center:<15} {hf.value:<25} {continuous}")

print()
print("Key: Only U(1) has a CONTINUOUS higher-form symmetry.")
print("     All non-Abelian groups have at most DISCRETE center symmetries.")

## 3. Goldstone vs Domain Wall

The Goldstone theorem tells us that spontaneous breaking of a **continuous** symmetry
produces gapless Nambu-Goldstone bosons -- these are the hydrodynamic modes.

But spontaneous breaking of a **discrete** symmetry produces something entirely
different: **domain walls** (topological defects). Domain walls are extended objects,
not propagating modes. They do not contribute to the hydrodynamic spectrum.

This distinction is the mechanism by which the algebraic constraint (higher-form
symmetries must be Abelian) translates into a physical consequence (no non-Abelian
gauge hydrodynamic modes):

- **Continuous 1-form (U(1)):** Breaking produces 1-form Goldstone -> photon
- **Discrete 1-form (Z_N):** Breaking produces domain walls -> no hydro mode
- **No higher-form:** No mechanism -> erasure

In [ ]:
# ════════════════════════════════════════════════════════════════════
# Goldstone vs domain wall for each higher-form symmetry type
# ════════════════════════════════════════════════════════════════════
symmetry_types = [
    HigherFormSymmetry.CONTINUOUS_1FORM,
    HigherFormSymmetry.DISCRETE_1FORM,
    HigherFormSymmetry.NONE,
]

print(f"{'Higher-form symmetry':<25} {'SSB product':<20} {'Hydrodynamic mode?'}")
print("-" * 65)
for sym in symmetry_types:
    fate = goldstone_or_domain_wall(sym)
    has_mode = fate in (HydrodynamicFate.GOLDSTONE_BOSON, HydrodynamicFate.SURVIVAL)
    print(f"{sym.value:<25} {fate.value:<20} {has_mode}")

print()
print("Applied to specific gauge groups:")
print("-" * 65)
for label, g in groups:
    hf = higher_form_symmetry_type(g)
    fate = goldstone_or_domain_wall(hf)
    survives = survives_hydrodynamization(g)
    symbol = "SURVIVES" if survives else "ERASED"
    print(f"  {label:<10} -> {hf.value:<25} -> {fate.value:<20} -> {symbol}")

## 4. The Erasure Theorem

**Theorem (Gauge Erasure):** A gauge group G produces surviving hydrodynamic
modes if and only if G is Abelian.

Equivalently: `survives_hydrodynamization(G) = is_abelian(G)`.

This is a sharp dichotomy with no exceptions. We verify it for all standard
gauge groups used in physics.

In [ ]:
# ════════════════════════════════════════════════════════════════════
# The erasure theorem: survival iff Abelian
# ════════════════════════════════════════════════════════════════════
test_groups = [
    u1(),
    su(2), su(3), su(4), su(5), su(6), su(7), su(8),
    so(3), so(5), so(10),
]

print("GAUGE ERASURE THEOREM VERIFICATION")
print("=" * 60)
print(f"{'Group':<10} {'Abelian':<10} {'Survives':<10} {'Match?'}")
print("-" * 40)

all_match = True
for g in test_groups:
    ab = is_abelian(g)
    sv = survives_hydrodynamization(g)
    match = (ab == sv)
    all_match = all_match and match
    status = "OK" if match else "FAIL"
    print(f"{g.name:<10} {str(ab):<10} {str(sv):<10} {status}")

print("-" * 40)
print(f"Theorem verified for all {len(test_groups)} groups: {'PASS' if all_match else 'FAIL'}")
print()
print("The dichotomy is exact: survives_hydrodynamization(G) = is_abelian(G)")

## 5. Standard Model Application

The Standard Model gauge group is SU(3)_c x SU(2)_L x U(1)_Y.

Applying the erasure theorem to each factor:
- **SU(3)_c (QCD):** Non-Abelian -> Z_3 center -> domain walls -> **ERASED**
- **SU(2)_L (weak):** Non-Abelian -> Z_2 center -> domain walls -> **ERASED**
- **U(1)_Y (hypercharge):** Abelian -> continuous 1-form -> Goldstone -> **SURVIVES**

After electroweak symmetry breaking (SU(2)_L x U(1)_Y -> U(1)_EM):
only U(1)_EM survives as a hydrodynamic mode (the photon).

In [ ]:
# ════════════════════════════════════════════════════════════════════
# Standard Model scorecard
# ════════════════════════════════════════════════════════════════════
sm_results = standard_model_analysis()

print("STANDARD MODEL GAUGE ERASURE SCORECARD")
print("=" * 70)
print(f"{'Factor':<15} {'Group':<10} {'Abelian':<10} {'Higher-form':<20} {'Fate':<15} {'Status'}")
print("-" * 70)

for factor_name, result in sm_results.items():
    status = "SURVIVES" if result.survives else "ERASED"
    print(
        f"{factor_name:<15} {result.group.name:<10} "
        f"{str(result.is_abelian):<10} {result.higher_form_type.value:<20} "
        f"{result.fate.value:<15} {status}"
    )

print()
print("Result: Only U(1) survives hydrodynamization.")
print("The photon is the unique gauge boson with a hydrodynamic analog.")

In [ ]:
# viz-ref: fig_sm_scorecard
# ════════════════════════════════════════════════════════════════════
# Standard Model scorecard — visual
# ════════════════════════════════════════════════════════════════════

sm_results = standard_model_analysis()

factor_names = list(sm_results.keys())
group_names = [r.group.name for r in sm_results.values()]
survives_list = [r.survives for r in sm_results.values()]

# Map each SM factor to its designated color
color_map = {
    "SU(3)_c": COLORS["Steinhauer"],
    "SU(2)_L": COLORS["Heidelberg"],
    "U(1)_Y": COLORS["Trento"],
}

bar_colors = [color_map.get(name, COLORS["cross"]) for name in factor_names]
bar_values = [1.0 if s else 0.0 for s in survives_list]
bar_text = ["SURVIVES" if s else "ERASED" for s in survives_list]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=factor_names,
    y=bar_values,
    text=bar_text,
    textposition="inside",
    textfont=dict(size=16, color="white", family=FONT["family"]),
    marker_color=bar_colors,
    marker_line=dict(width=2, color="black"),
    width=0.5,
))

# Add group name annotations below bars
for i, (factor, group) in enumerate(zip(factor_names, group_names)):
    fig.add_annotation(
        x=factor, y=-0.15,
        text=group,
        showarrow=False,
        font=dict(size=13, family=FONT["family"]),
        yref="paper",
    )

apply_layout(
    fig,
    title=dict(text="Standard Model Gauge Erasure Scorecard", font=TITLE_FONT),
    yaxis=dict(
        tickvals=[0, 1],
        ticktext=["Erased", "Survives"],
        range=[-0.1, 1.3],
        showgrid=False,
    ),
    xaxis=dict(title="Standard Model gauge factor"),
    height=400,
    width=550,
    showlegend=False,
)

fig.show()

## 6. N=4 SYM: The Decisive Test

N=4 Super-Yang-Mills theory provides the decisive test case.

**Why it matters:** N=4 SYM is:
- **Non-confining** at all coupling strengths (it is exactly conformal)
- Color-charged states exist in the spectrum at all couplings
- Yet holographic hydrodynamics (via AdS/CFT) involves exclusively color-singlet operators

**The argument from confinement would fail here:** if gauge erasure were caused by
confinement (color charges screened/confined), then N=4 SYM should NOT show
erasure. But it does.

**Our explanation works:** the erasure is structural, from the Abelian constraint
on higher-form symmetries. N=4 SYM with gauge group SU(N) still has only a
discrete Z_N center symmetry, so the erasure theorem applies regardless of
confinement.

In [ ]:
# ════════════════════════════════════════════════════════════════════
# N=4 SYM analysis
# ════════════════════════════════════════════════════════════════════
n4_result = n4_sym_analysis()

print("N=4 SUPER-YANG-MILLS: THE DECISIVE TEST")
print("=" * 70)
print(f"  Gauge group:        {n4_result.group.name}")
print(f"  Is Abelian:         {n4_result.is_abelian}")
print(f"  Is confining:       No (conformal at all couplings)")
print(f"  Higher-form type:   {n4_result.higher_form_type.value}")
print(f"  Hydrodynamic fate:  {n4_result.fate.value}")
print(f"  Survives hydro:     {n4_result.survives}")
print()
print("Explanation:")
print(f"  {n4_result.explanation}")
print()
print("Conclusion:")
print("  Erasure occurs even without confinement.")
print("  The mechanism is structural (higher-form Abelian constraint),")
print("  not dynamical (confinement of color charges).")

In [ ]:
# viz-ref: fig_erasure_survey
# ════════════════════════════════════════════════════════════════════
# Central figure: Gauge erasure survey across all groups
# ════════════════════════════════════════════════════════════════════

survey_groups = [
    ("U(1)", u1()),
    ("SU(2)", su(2)),
    ("SU(3)", su(3)),
    ("SU(4)\n(N=4 SYM)", su(4)),
    ("SU(5)", su(5)),
    ("SO(3)", so(3)),
    ("SO(5)", so(5)),
    ("SO(10)", so(10)),
]

names = [label for label, _ in survey_groups]
survives_vals = [1.0 if survives_hydrodynamization(g) else 0.0 for _, g in survey_groups]
colors = [
    COLORS["dispersive"] if survives_hydrodynamization(g)
    else COLORS["dissipative"]
    for _, g in survey_groups
]
labels = ["SURVIVES" if survives_hydrodynamization(g) else "ERASED" for _, g in survey_groups]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=names,
    y=survives_vals,
    text=labels,
    textposition="inside",
    textfont=dict(size=14, color="white", family=FONT["family"]),
    marker_color=colors,
    marker_line=dict(width=2, color="black"),
    width=0.6,
))

# Add center subgroup annotations
for i, (label, g) in enumerate(survey_groups):
    fig.add_annotation(
        x=label, y=0.5,
        text=f"Center: {g.center}",
        showarrow=False,
        font=dict(size=10, color="white", family=FONT["family"]),
    )

apply_layout(
    fig,
    title=dict(
        text="Gauge Erasure Theorem: Survival iff Abelian",
        font=TITLE_FONT,
    ),
    yaxis=dict(
        tickvals=[0, 1],
        ticktext=["Erased", "Survives"],
        range=[-0.1, 1.3],
        showgrid=False,
    ),
    xaxis=dict(title="Gauge group"),
    height=450,
    width=700,
    showlegend=False,
)

# Add legend-like annotations
c_surv = COLORS["dispersive"]
c_eras = COLORS["dissipative"]
fig.add_annotation(
    x=0.98, y=0.95, xref="paper", yref="paper",
    text=(
        f"<span style='color:{c_surv}'>\u25a0</span> Abelian (survives)  "
        f"<span style='color:{c_eras}'>\u25a0</span> Non-Abelian (erased)"
    ),
    showarrow=False,
    font=dict(size=12, family=FONT["family"]),
    align="right",
    xanchor="right",
)

fig.show()

## 7. Universality Across Frameworks

The gauge erasure theorem holds universally across all known hydrodynamic frameworks:

### Standard (Mueller-Israel-Stewart) Hydrodynamics
The standard formulation builds constitutive relations from conserved currents.
Only 0-form conserved charges (energy, momentum, particle number) and the
1-form Goldstone from U(1) enter. Non-Abelian gauge fields have no conserved
current in the hydrodynamic sense (covariant conservation != ordinary conservation).

### Fracton Hydrodynamics
Fracton phases feature restricted mobility due to higher-multipole conservation
(e.g., dipole conservation). One might hope that the additional conservation
laws could stabilize non-Abelian gauge modes. But color charges are
*covariantly* conserved, not *ordinarily* conserved -- fracton constraints
require the latter. The erasure persists.

### Holographic Hydrodynamics
In the holographic (AdS/CFT) framework, the hydrodynamic modes are dual to
quasinormal modes of black branes in the bulk. For N=4 SYM, the holographic
fluid involves exclusively color-singlet operators (stress tensor, R-symmetry
currents). This is the gauge erasure in action: the holographic limit
automatically projects onto the color-singlet sector.

## 8. Lean Verification

The gauge erasure theorem has been formalized in Lean 4 (`GaugeErasure.lean`).

**11 Theorems + 1 Axiom:**

| # | Lean Name | Statement |
|---|-----------|----------|
| 1 | `is_abelian_def` | Definition of Abelian gauge group |
| 2 | `center_subgroup` | Center subgroup classification |
| 3 | `higher_form_abelian` | Higher-form symmetries must be Abelian |
| 4 | `goldstone_requires_continuous` | Goldstone theorem requires continuous symmetry |
| 5 | `discrete_gives_domain_wall` | Discrete SSB -> domain walls |
| 6 | `no_nonabelian_goldstone` | No Goldstone from non-Abelian higher-form |
| 7 | `gauge_erasure_theorem` | Main theorem: survival iff Abelian |
| 8 | `u1_survives` | U(1) produces photon as 1-form Goldstone |
| 9 | `su_erased` | SU(N) gauge DOF erased for all N >= 2 |
| 10 | `so_erased` | SO(N) gauge DOF erased for all N >= 3 |
| 11 | `sm_scorecard` | Standard Model: only U(1) survives |
| A1 | `higher_form_commutativity` | Axiom: codimension >1 operators commute |

In [ ]:
# ════════════════════════════════════════════════════════════════════
# Lean theorem verification status
# ════════════════════════════════════════════════════════════════════
lean_theorems = [
    ("is_abelian_def", "Definition of Abelian gauge group", "theorem"),
    ("center_subgroup", "Center subgroup classification", "theorem"),
    ("higher_form_abelian", "Higher-form symmetries must be Abelian", "theorem"),
    ("goldstone_requires_continuous", "Goldstone theorem requires continuous symmetry", "theorem"),
    ("discrete_gives_domain_wall", "Discrete SSB -> domain walls", "theorem"),
    ("no_nonabelian_goldstone", "No Goldstone from non-Abelian higher-form", "theorem"),
    ("gauge_erasure_theorem", "Main theorem: survival iff Abelian", "theorem"),
    ("u1_survives", "U(1) produces photon as 1-form Goldstone", "theorem"),
    ("su_erased", "SU(N) erased for all N >= 2", "theorem"),
    ("so_erased", "SO(N) erased for all N >= 3", "theorem"),
    ("sm_scorecard", "Standard Model: only U(1) survives", "theorem"),
    ("higher_form_commutativity", "Codimension >1 operators commute", "axiom"),
]

print("LEAN 4 FORMALIZATION: GaugeErasure.lean")
print("=" * 70)
print(f"{'#':<4} {'Type':<10} {'Lean Name':<30} {'Statement'}")
print("-" * 70)
for i, (name, desc, kind) in enumerate(lean_theorems, 1):
    label = "A1" if kind == "axiom" else str(i)
    print(f"{label:<4} {kind:<10} {name:<30} {desc}")

n_theorems = sum(1 for _, _, k in lean_theorems if k == "theorem")
n_axioms = sum(1 for _, _, k in lean_theorems if k == "axiom")
print(f"\nTotal: {n_theorems} theorems + {n_axioms} axiom")
print("The single axiom (higher_form_commutativity) encodes the topological input.")
print("All theorems follow deductively from this axiom + standard group theory.")

## 9. Implications for Analog Gravity

The gauge erasure theorem has direct implications for the analog gravity program
and emergent physics more broadly:

### What emerges
- **Electrodynamics (U(1)):** The photon can emerge as a 1-form Goldstone boson
  in a fluid or condensed matter system. This is the content of the
  Grozdanov-Hofman-Iqbal photonization theorem.
- **Gravity (metric):** Acoustic metrics in BEC systems provide analog Hawking
  radiation, as studied in Papers 1 and 2 of this series.

### What cannot emerge
- **QCD (SU(3)):** Color charge cannot survive hydrodynamization. No fluid
  system can carry non-Abelian gauge degrees of freedom.
- **Weak force (SU(2)):** Same structural obstruction.
- **Any non-Abelian gauge theory:** The result is universal.

### The boundary of emergence
This draws a sharp line through the emergent physics program:
- Gravity + electrodynamics: accessible via fluid analogs
- Non-Abelian gauge theories: structurally inaccessible

This is not a practical limitation ("we haven't found the right fluid") but a
theorem: the algebraic structure of hydrodynamics is incompatible with
non-Abelian gauge symmetry.